# Day 53：LLaMA-Factory LoRA 微调实战（配置生成、参数数学与 Loss 曲线诊断）

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第五阶段（Stage 5：微调与部署）的 **Day 53**！今天我们将完成 **LoRA 微调配置生成（`LoRATrainingConfig` / `export_llamafactory_yaml`）**、**可训练参数量精确核算（`count_lora_trainable_params`）** 以及 **训练仿真与过拟合/欠拟合诊断（`simulate_training_run`）**。

## 1. 环境初始化与导入 `project_iterations/day_53` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day53_dir = repo_root / "project_iterations" / "day_53"
if str(day53_dir) not in sys.path:
    sys.path.insert(0, str(day53_dir))

from zhishu_agent import (
    LoRATrainingConfig,
    count_lora_trainable_params,
    export_llamafactory_yaml,
    simulate_training_run,
)

print("已加载模块路径:", day53_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_53


## 2. 构建 `LoRATrainingConfig` 并导出 LLaMA-Factory YAML

In [2]:
config = LoRATrainingConfig(
    model_name_or_path="Qwen/Qwen2.5-7B-Instruct",
    finetuning_type="lora",
    lora_rank=8,
    lora_alpha=16,
    learning_rate=1e-4,
    num_train_epochs=3.0,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    template="qwen",
)

print("LoRA Scaling (alpha/rank):", config.lora_scaling)
print("单卡 Effective Batch Size:", config.effective_batch_size(num_gpus=1))
print("4卡 Effective Batch Size:", config.effective_batch_size(num_gpus=4))
print("\n--- 导出的 LLaMA-Factory YAML ---")
print(export_llamafactory_yaml(config))

LoRA Scaling (alpha/rank): 2.0
单卡 Effective Batch Size: 16
4卡 Effective Batch Size: 64

--- 导出的 LLaMA-Factory YAML ---
### model
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 8
lora_alpha: 16
lora_target: all

### dataset
dataset: zhishu_enterprise_sft
template: qwen
cutoff_len: 2048
preprocessing_num_workers: 4

### output
output_dir: saves/qwen2.5-7b/lora/zhishu-sft
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 4
gradient_accumulation_steps: 4
learning_rate: 1.0e-04
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true

### eval
val_size: 0.1
per_device_eval_batch_size: 2
eval_strategy: steps
eval_steps: 50



## 3. LoRA 可训练参数数学核算（以 Qwen2.5-7B 为例）

In [3]:
for r in (4, 8, 16, 32):
    stats = count_lora_trainable_params(
        hidden_size=3584,
        num_hidden_layers=28,
        lora_rank=r,
        target_modules=("q_proj", "k_proj", "v_proj", "o_proj"),
        base_model_params=7_615_616_512,
    )
    print(
        f"rank={r:2d} | trainable={stats.trainable_params:>10,} | "
        f"ratio={stats.trainable_percentage:.4f}% | fp16_mem={stats.lora_memory_fp16_mb:.2f} MB"
    )

rank= 4 | trainable= 3,211,264 | ratio=0.0422% | fp16_mem=6.12 MB
rank= 8 | trainable= 6,422,528 | ratio=0.0843% | fp16_mem=12.25 MB
rank=16 | trainable=12,845,056 | ratio=0.1687% | fp16_mem=24.50 MB
rank=32 | trainable=25,690,112 | ratio=0.3373% | fp16_mem=49.00 MB


## 4. 模拟训练曲线与过拟合/欠拟合诊断

In [4]:
scenarios = [
    ("健康收敛", LoRATrainingConfig(lora_rank=8, lora_alpha=16, learning_rate=1e-4, num_train_epochs=3.0), 1000),
    ("欠拟合", LoRATrainingConfig(lora_rank=2, lora_alpha=4, learning_rate=1e-6, num_train_epochs=0.5), 1000),
    ("过拟合", LoRATrainingConfig(lora_rank=64, lora_alpha=128, learning_rate=2e-4, num_train_epochs=10.0), 200),
]

for label, cfg, ds_size in scenarios:
    res = simulate_training_run(cfg, dataset_size=ds_size)
    print(f"[{label}] diagnosis={res.diagnosis} | train={res.final_train_loss:.4f} | "
          f"eval={res.final_eval_loss:.4f} | best_eval={res.best_eval_loss:.4f} (step {res.best_step}) | "
          f"gap={res.generalization_gap:.4f}")

[健康收敛] diagnosis=healthy | train=0.5896 | eval=0.7096 | best_eval=0.7096 (step 189) | gap=0.1200
[欠拟合] diagnosis=underfitting | train=2.3682 | eval=2.4882 | best_eval=2.4882 (step 32) | gap=0.1200
[过拟合] diagnosis=overfitting | train=0.1800 | eval=2.0002 | best_eval=0.2849 (step 52) | gap=1.8202
